In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os

base = '/content/drive/MyDrive/Pandemic_Dashboard_Project'

folders = [
    f'{base}/data/raw',
    f'{base}/data/cleaned',
    f'{base}/data/sql',
    f'{base}/visuals',
    f'{base}/excel_exports',
    f'{base}/reports'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f'Created: {folder}')

print('\nAll folders ready!')

Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/data/raw
Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/data/cleaned
Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/data/sql
Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/visuals
Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/excel_exports
Created: /content/drive/MyDrive/Pandemic_Dashboard_Project/reports

All folders ready!


In [4]:

!pip install openpyxl xlsxwriter --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import sqlite3
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)

print("All libraries loaded successfully!")
print(f"Pandas  version : {pd.__version__}")
print(f"NumPy   version : {np.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 6.4 MB/s eta 0:00:00
All libraries loaded successfully!
Pandas  version : 2.2.2
NumPy   version : 2.0.2


Data Collection

In [5]:
base_url = (
    "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/"
    "csse_covid_19_data/csse_covid_19_time_series/"
)

sources = {
    'confirmed' : base_url + "time_series_covid19_confirmed_global.csv",
    'deaths'    : base_url + "time_series_covid19_deaths_global.csv",
    'recovered' : base_url + "time_series_covid19_recovered_global.csv",
}

raw = {}
for name, url in sources.items():
    raw[name] = pd.read_csv(url)
    save_path = f'{base}/data/raw/{name}_raw.csv'
    raw[name].to_csv(save_path, index=False)
    print(f"  {name:12s} → {raw[name].shape[0]:>4} rows x {raw[name].shape[1]:>4} columns  [saved]")

print("\nRaw data download complete!")

  confirmed    →  289 rows x 1147 columns  [saved]
  deaths       →  289 rows x 1147 columns  [saved]
  recovered    →  274 rows x 1147 columns  [saved]

Raw data download complete!


In [6]:
print("=" * 55)
print("  CONFIRMED DATASET — STRUCTURE PREVIEW")
print("=" * 55)

print(f"\nShape      : {raw['confirmed'].shape}")
print(f"\nColumns (first 8) : {list(raw['confirmed'].columns[:8])}")
print(f"\nColumns (last 3)  : {list(raw['confirmed'].columns[-3:])}")
print(f"\nUnique countries  : {raw['confirmed']['Country/Region'].nunique()}")
print(f"\nSample rows:")
print(raw['confirmed'][['Province/State','Country/Region','Lat','Long']].head(6))

  CONFIRMED DATASET — STRUCTURE PREVIEW

Shape      : (289, 1147)

Columns (first 8) : ['Province/State', 'Country/Region', 'Lat', 'Long', '1/22/20', '1/23/20', '1/24/20', '1/25/20']

Columns (last 3)  : ['3/7/23', '3/8/23', '3/9/23']

Unique countries  : 201

Sample rows:
  Province/State Country/Region    Lat  Long
0            NaN    Afghanistan  33.94 67.71
1            NaN        Albania  41.15 20.17
2            NaN        Algeria  28.03  1.66
3            NaN        Andorra  42.51  1.52
4            NaN         Angola -11.20 17.87
5            NaN     Antarctica -71.95 23.35


Data Cleaning and Transformation

In [7]:
def clean_and_melt(df, value_name):
    # keeping only country + dates
    df = df.drop(columns=['Province/State', 'Lat', 'Long'], errors='ignore')

    # Collapse province-level rows into single country rows
    df = df.groupby('Country/Region').sum(numeric_only=True).reset_index()

    # Convert from wide format → long format
    df = df.melt(id_vars=['Country/Region'], var_name='Date', value_name=value_name)

    # Parse date strings into proper datetime
    df['Date'] = pd.to_datetime(df['Date'])

    return df

confirmed_clean = clean_and_melt(raw['confirmed'], 'Confirmed')
deaths_clean    = clean_and_melt(raw['deaths'],    'Deaths')
recovered_clean = clean_and_melt(raw['recovered'], 'Recovered')

print("Shapes after cleaning:")
print(f"  Confirmed : {confirmed_clean.shape}")
print(f"  Deaths    : {deaths_clean.shape}")
print(f"  Recovered : {recovered_clean.shape}")

print(f"\nSample (Confirmed):")
print(confirmed_clean[confirmed_clean['Country/Region'] == 'India'].tail(4))

Shapes after cleaning:
  Confirmed : (229743, 3)
  Deaths    : (229743, 3)
  Recovered : (229743, 3)

Sample (Confirmed):
       Country/Region       Date  Confirmed
229019          India 2023-03-06   44689593
229220          India 2023-03-07   44689919
229421          India 2023-03-08   44690298
229622          India 2023-03-09   44690738


In [8]:
# Merge all three on Country + Date
df = confirmed_clean.merge(deaths_clean,    on=['Country/Region', 'Date'])
df = df.merge(recovered_clean,              on=['Country/Region', 'Date'])

# Rename it
df = df.rename(columns={'Country/Region': 'Country'})
df = df.sort_values(['Country', 'Date']).reset_index(drop=True)

# Add derived columns
df['Active']        = (df['Confirmed'] - df['Deaths'] - df['Recovered']).clip(lower=0)
df['Fatality_Rate'] = (df['Deaths']    / df['Confirmed'].replace(0, np.nan) * 100).round(4)
df['Recovery_Rate'] = (df['Recovered'] / df['Confirmed'].replace(0, np.nan) * 100).round(4)

# Add time columns
df['Year']       = df['Date'].dt.year
df['Month']      = df['Date'].dt.month
df['Month_Name'] = df['Date'].dt.strftime('%b')
df['YearMonth']  = df['Date'].dt.to_period('M').astype(str)

# Saving cleaned file
df.to_csv(f'{base}/data/cleaned/covid_master.csv', index=False)

print("=" * 50)
print("  MASTER DATASET CREATED")
print("=" * 50)
print(f"  Total rows       : {len(df):,}")
print(f"  Total columns    : {df.shape[1]}")
print(f"  Countries        : {df['Country'].nunique()}")
print(f"  Date range       : {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"  Total days       : {df['Date'].nunique()}")
print(f"\n  Columns created  : {list(df.columns)}")
print(f"\n  Sample rows:")
print(df.head(4).to_string())

  MASTER DATASET CREATED
  Total rows       : 229,743
  Total columns    : 12
  Countries        : 201
  Date range       : 2020-01-22 → 2023-03-09
  Total days       : 1143

  Columns created  : ['Country', 'Date', 'Confirmed', 'Deaths', 'Recovered', 'Active', 'Fatality_Rate', 'Recovery_Rate', 'Year', 'Month', 'Month_Name', 'YearMonth']

  Sample rows:
       Country       Date  Confirmed  Deaths  Recovered  Active  Fatality_Rate  Recovery_Rate  Year  Month Month_Name YearMonth
0  Afghanistan 2020-01-22          0       0          0       0            NaN            NaN  2020      1        Jan   2020-01
1  Afghanistan 2020-01-23          0       0          0       0            NaN            NaN  2020      1        Jan   2020-01
2  Afghanistan 2020-01-24          0       0          0       0            NaN            NaN  2020      1        Jan   2020-01
3  Afghanistan 2020-01-25          0       0          0       0            NaN            NaN  2020      1        Jan   2020-01


Data Quality Analysis

In [9]:


issues_log = []

# ── FINDING 1: Null / Missing Values ──────────────────────────
print("\n[ FINDING 1 ] NULL & MISSING VALUES")
print("-" * 45)

null_counts = df.isnull().sum()
null_pct    = (null_counts / len(df) * 100).round(2)

for col in df.columns:
    if null_counts[col] > 0:
        print(f"  {col:<20} : {null_counts[col]:>8,} nulls  ({null_pct[col]}%)")
        issues_log.append({
            'Finding_No'  : 1,
            'Category'    : 'Null Values',
            'Column'      : col,
            'Issue_Count' : null_counts[col],
            'Issue_%'     : null_pct[col],
            'Detail'      : f"{null_counts[col]:,} null values in {col}"
        })

if null_counts.sum() == 0:
    print("  No nulls found in numeric columns — PASS")


[ FINDING 1 ] NULL & MISSING VALUES
---------------------------------------------
  Fatality_Rate        :   15,630 nulls  (6.8%)
  Recovery_Rate        :   15,630 nulls  (6.8%)


In [10]:
# ── FINDING 2: Zero-Value Anomalies ───────────────────────────
print("\n[ FINDING 2 ] ZERO-VALUE ANOMALY DETECTION")
print("-" * 45)

latest = df[df['Date'] == df['Date'].max()].copy()

# Countries with confirmed > 10,000 but zero deaths
zero_deaths = latest[
    (latest['Confirmed'] > 10000) & (latest['Deaths'] == 0)
]['Country'].tolist()

# Countries with confirmed > 1,000 but zero recoveries
zero_recovered = latest[
    (latest['Confirmed'] > 1000) & (latest['Recovered'] == 0)
]['Country'].tolist()

print(f"  Countries with >10k confirmed but 0 deaths     : {len(zero_deaths)}")
print(f"  → {zero_deaths}")

print(f"\n  Countries with >1k confirmed but 0 recoveries  : {len(zero_recovered)}")
print(f"  → {zero_recovered[:10]}")

if zero_deaths:
    issues_log.append({
        'Finding_No'  : 2,
        'Category'    : 'Zero Death Anomaly',
        'Column'      : 'Deaths',
        'Issue_Count' : len(zero_deaths),
        'Issue_%'     : round(len(zero_deaths)/latest['Country'].nunique()*100, 2),
        'Detail'      : f"Countries flagged: {zero_deaths}"
    })

if zero_recovered:
    issues_log.append({
        'Finding_No'  : 2,
        'Category'    : 'Zero Recovery Anomaly',
        'Column'      : 'Recovered',
        'Issue_Count' : len(zero_recovered),
        'Issue_%'     : round(len(zero_recovered)/latest['Country'].nunique()*100, 2),
        'Detail'      : f"{len(zero_recovered)} countries with 0 recoveries"
    })


[ FINDING 2 ] ZERO-VALUE ANOMALY DETECTION
---------------------------------------------
  Countries with >10k confirmed but 0 deaths     : 0
  → []

  Countries with >1k confirmed but 0 recoveries  : 194
  → ['Afghanistan', 'Albania', 'Algeria', 'Andorra', 'Angola', 'Antigua and Barbuda', 'Argentina', 'Armenia', 'Australia', 'Austria']


In [11]:
# ── FINDING 3: Duplicate Records ──────────────────────────────
print("\n[ FINDING 3 ] DUPLICATE RECORDS")
print("-" * 45)

dupes = df.duplicated(subset=['Country', 'Date']).sum()
print(f"  Duplicate (Country + Date) pairs : {dupes}")

if dupes > 0:
    print(df[df.duplicated(subset=['Country','Date'], keep=False)].head(5))
    issues_log.append({
        'Finding_No'  : 3,
        'Category'    : 'Duplicate Records',
        'Column'      : 'Country + Date',
        'Issue_Count' : dupes,
        'Issue_%'     : round(dupes/len(df)*100, 4),
        'Detail'      : f"{dupes} duplicate country+date combinations"
    })
else:
    print("  No duplicates found — PASS")

# ── FINDING 4: Countries That Stopped Reporting ───────────────
print("\n[ FINDING 4 ] INCONSISTENT / STOPPED REPORTING")
print("-" * 45)

max_date          = df['Date'].max()
last_report       = df.groupby('Country')['Date'].max().reset_index()
last_report.columns = ['Country', 'Last_Report_Date']
last_report['Days_Silent'] = (max_date - last_report['Last_Report_Date']).dt.days
stopped           = last_report[last_report['Days_Silent'] > 60].sort_values('Days_Silent', ascending=False)

print(f"  Countries silent >60 days before dataset end : {len(stopped)}")
print(stopped.head(10).to_string(index=False))

if len(stopped) > 0:
    issues_log.append({
        'Finding_No'  : 4,
        'Category'    : 'Stopped Reporting',
        'Column'      : 'Date',
        'Issue_Count' : len(stopped),
        'Issue_%'     : round(len(stopped)/df['Country'].nunique()*100, 2),
        'Detail'      : f"{len(stopped)} countries stopped reporting >60 days early"
    })


[ FINDING 3 ] DUPLICATE RECORDS
---------------------------------------------
  Duplicate (Country + Date) pairs : 0
  No duplicates found — PASS

[ FINDING 4 ] INCONSISTENT / STOPPED REPORTING
---------------------------------------------
  Countries silent >60 days before dataset end : 0
Empty DataFrame
Columns: [Country, Last_Report_Date, Days_Silent]
Index: []


In [12]:
# ── FINDING 5: Statistical Outliers ───────────────────────────
print("\n[ FINDING 5 ] STATISTICAL OUTLIER DETECTION")
print("-" * 45)

global_daily          = df.groupby('Date')['Confirmed'].sum().reset_index()
global_daily['Daily_New'] = global_daily['Confirmed'].diff().clip(lower=0)

mean_d     = global_daily['Daily_New'].mean()
std_d      = global_daily['Daily_New'].std()
threshold  = mean_d + 4 * std_d
outliers   = global_daily[global_daily['Daily_New'] > threshold]

print(f"  Mean daily new cases      : {mean_d:>12,.0f}")
print(f"  Std deviation             : {std_d:>12,.0f}")
print(f"  4-sigma threshold         : {threshold:>12,.0f}")
print(f"  Outlier days detected     : {len(outliers)}")
print(outliers[['Date','Daily_New']].to_string(index=False))

if len(outliers) > 0:
    issues_log.append({
        'Finding_No'  : 5,
        'Category'    : 'Statistical Outliers',
        'Column'      : 'Daily_New_Cases',
        'Issue_Count' : len(outliers),
        'Issue_%'     : round(len(outliers)/global_daily.shape[0]*100, 2),
        'Detail'      : f"{len(outliers)} days exceeded 4-sigma threshold"
    })

# ── FINDING 6: Data Completeness Score ────────────────────────
print("\n[ FINDING 6 ] DATA COMPLETENESS SCORE")
print("-" * 45)

expected_days        = df['Date'].nunique()
completeness         = df.groupby('Country')['Date'].count().reset_index()
completeness.columns = ['Country', 'Reported_Days']
completeness['Expected_Days']      = expected_days
completeness['Completeness_%']     = (completeness['Reported_Days'] / expected_days * 100).round(2)

full_coverage   = (completeness['Completeness_%'] == 100).sum()
avg_completeness= completeness['Completeness_%'].mean()

print(f"  Expected days per country : {expected_days}")
print(f"  Countries with 100%       : {full_coverage} / {len(completeness)}")
print(f"  Average completeness      : {avg_completeness:.2f}%")
print(f"\n  Lowest completeness countries:")
print(completeness.nsmallest(8, 'Completeness_%').to_string(index=False))

# ── Saving the findings ────────────────────────────────────────
import os
os.makedirs(f'{base}/data/quality_reports', exist_ok=True)
issues_df = pd.DataFrame(issues_log)
issues_df.to_csv(f'{base}/data/quality_reports/data_quality_issues.csv', index=False)
completeness.to_csv(f'{base}/data/quality_reports/completeness_scores.csv', index=False)

print("\n" + "=" * 60)
print(f"  TOTAL ISSUES LOGGED : {len(issues_log)}")
print(f"  Quality report saved to data/quality_reports/")
print("=" * 60)


[ FINDING 5 ] STATISTICAL OUTLIER DETECTION
---------------------------------------------
  Mean daily new cases      :      592,443
  Std deviation             :      595,651
  4-sigma threshold         :    2,975,047
  Outlier days detected     : 18
      Date    Daily_New
2022-01-10 3,251,715.00
2022-01-11 3,030,407.00
2022-01-12 3,446,654.00
2022-01-13 3,187,105.00
2022-01-14 3,316,453.00
2022-01-18 3,764,604.00
2022-01-19 4,083,281.00
2022-01-20 3,734,294.00
2022-01-21 3,810,117.00
2022-01-24 3,632,860.00
2022-01-25 3,653,814.00
2022-01-26 3,761,673.00
2022-01-27 3,699,732.00
2022-01-28 3,678,929.00
2022-01-31 3,859,475.00
2022-02-01 3,129,732.00
2022-02-02 3,184,645.00
2022-02-03 3,194,077.00

[ FINDING 6 ] DATA COMPLETENESS SCORE
---------------------------------------------
  Expected days per country : 1143
  Countries with 100%       : 201 / 201
  Average completeness      : 100.00%

  Lowest completeness countries:
            Country  Reported_Days  Expected_Days  Complete

SQL Analysis

In [13]:
import sqlite3

# Create SQLite database and load master data
db_path = f'{base}/data/sql/pandemic.db'
conn    = sqlite3.connect(db_path)

df.to_sql('covid_data', conn, if_exists='replace', index=False)

print("Database created successfully!")
print(f"Saved at: {db_path}")

# Verify it loaded correctly
verify = pd.read_sql("SELECT COUNT(*) as total_rows, COUNT(DISTINCT Country) as countries FROM covid_data", conn)
print(f"\nDatabase verification:")
print(verify.to_string(index=False))

Database created successfully!
Saved at: /content/drive/MyDrive/Pandemic_Dashboard_Project/data/sql/pandemic.db

Database verification:
 total_rows  countries
     229743        201


In [14]:
query1 = """
SELECT
    MAX(Date)                                          AS Latest_Date,
    SUM(Confirmed)                                     AS Total_Confirmed,
    SUM(Deaths)                                        AS Total_Deaths,
    SUM(Recovered)                                     AS Total_Recovered,
    ROUND(SUM(Deaths) * 100.0 / SUM(Confirmed), 2)    AS Global_Fatality_Rate_Pct,
    ROUND(SUM(Recovered) * 100.0 / SUM(Confirmed), 2) AS Global_Recovery_Rate_Pct,
    COUNT(DISTINCT Country)                            AS Countries_Tracked
FROM covid_data
WHERE Date = (SELECT MAX(Date) FROM covid_data)
"""

kpi = pd.read_sql(query1, conn)
print("=" * 55)
print("  SQL QUERY 1 — GLOBAL KPI SUMMARY")
print("=" * 55)
print(kpi.T.to_string(header=False))
kpi.to_csv(f'{base}/data/sql/sql_query1_global_kpi.csv', index=False)
print("\nSaved: sql_query1_global_kpi.csv")

  SQL QUERY 1 — GLOBAL KPI SUMMARY
Latest_Date               2023-03-09 00:00:00
Total_Confirmed                     676570149
Total_Deaths                          6881802
Total_Recovered                             0
Global_Fatality_Rate_Pct                 1.02
Global_Recovery_Rate_Pct                 0.00
Countries_Tracked                         201

Saved: sql_query1_global_kpi.csv


In [15]:
query2 = """
SELECT
    Country,
    MAX(Confirmed)                                         AS Total_Confirmed,
    MAX(Deaths)                                            AS Total_Deaths,
    MAX(Recovered)                                         AS Total_Recovered,
    ROUND(MAX(Deaths) * 100.0 / NULLIF(MAX(Confirmed),0), 2)    AS Fatality_Rate_Pct,
    ROUND(MAX(Recovered) * 100.0 / NULLIF(MAX(Confirmed),0), 2) AS Recovery_Rate_Pct
FROM covid_data
GROUP BY Country
ORDER BY Total_Confirmed DESC
LIMIT 10
"""

top10 = pd.read_sql(query2, conn)
print("=" * 55)
print("  SQL QUERY 2 — TOP 10 COUNTRIES BY CONFIRMED CASES")
print("=" * 55)
print(top10.to_string(index=False))
top10.to_csv(f'{base}/data/sql/sql_query2_top10_countries.csv', index=False)
print("\nSaved: sql_query2_top10_countries.csv")

  SQL QUERY 2 — TOP 10 COUNTRIES BY CONFIRMED CASES
       Country  Total_Confirmed  Total_Deaths  Total_Recovered  Fatality_Rate_Pct  Recovery_Rate_Pct
            US        103802702       1123836          6298082               1.08               6.07
         India         44690738        530779         30974748               1.19              69.31
        France         39866718        166176           415111               0.42               1.04
       Germany         38249060        168935          3659260               0.44               9.57
        Brazil         37081209        699276         17771228               1.89              47.93
         Japan         33320438         72997           852451               0.22               2.56
  Korea, South         30615522         34093           180719               0.11               0.59
         Italy         25603510        188322          4144608               0.74              16.19
United Kingdom         24658705        

In [16]:
query3 = """
SELECT
    Year,
    ROUND(MAX(Confirmed) - MIN(Confirmed), 0)   AS New_Cases_That_Year,
    ROUND(MAX(Deaths)    - MIN(Deaths),    0)   AS New_Deaths_That_Year,
    ROUND(MAX(Recovered) - MIN(Recovered), 0)   AS New_Recovered_That_Year,
    COUNT(DISTINCT Country)                     AS Countries_Reporting
FROM covid_data
GROUP BY Year
ORDER BY Year
"""

yearly = pd.read_sql(query3, conn)
print("=" * 55)
print("  SQL QUERY 3 — YEARLY GLOBAL TREND")
print("=" * 55)
print(yearly.to_string(index=False))
yearly.to_csv(f'{base}/data/sql/sql_query3_yearly_trend.csv', index=False)
print("\nSaved: sql_query3_yearly_trend.csv")

  SQL QUERY 3 — YEARLY GLOBAL TREND
 Year  New_Cases_That_Year  New_Deaths_That_Year  New_Recovered_That_Year  Countries_Reporting
 2020        20,219,878.00            350,604.00             9,883,461.00                  201
 2021        54,907,717.00            825,468.00            30,974,748.00                  201
 2022       100,765,333.00          1,092,764.00                     1.00                  201
 2023       103,802,701.00          1,123,836.00                     0.00                  201

Saved: sql_query3_yearly_trend.csv


In [17]:
query4 = """
SELECT
    Country,
    MAX(Confirmed)  AS Total_Confirmed,
    MAX(Deaths)     AS Total_Deaths,
    MAX(Recovered)  AS Total_Recovered,
    CASE
        WHEN MAX(Recovered) = 0 AND MAX(Confirmed) > 1000
        THEN 'FLAGGED — Zero Recovery Anomaly'
        ELSE 'Normal'
    END AS Data_Quality_Flag
FROM covid_data
WHERE Date = (SELECT MAX(Date) FROM covid_data)
GROUP BY Country
HAVING Data_Quality_Flag = 'FLAGGED — Zero Recovery Anomaly'
ORDER BY Total_Confirmed DESC
LIMIT 15
"""

anomaly = pd.read_sql(query4, conn)
print("=" * 55)
print("  SQL QUERY 4 — DATA QUALITY FLAGS (ZERO RECOVERY)")
print("=" * 55)
print(anomaly.to_string(index=False))
anomaly.to_csv(f'{base}/data/sql/sql_query4_anomaly_flags.csv', index=False)
print("\nSaved: sql_query4_anomaly_flags.csv")

  SQL QUERY 4 — DATA QUALITY FLAGS (ZERO RECOVERY)
       Country  Total_Confirmed  Total_Deaths  Total_Recovered               Data_Quality_Flag
            US        103802702       1123836                0 FLAGGED — Zero Recovery Anomaly
         India         44690738        530779                0 FLAGGED — Zero Recovery Anomaly
        France         39866718        166176                0 FLAGGED — Zero Recovery Anomaly
       Germany         38249060        168935                0 FLAGGED — Zero Recovery Anomaly
        Brazil         37076053        699276                0 FLAGGED — Zero Recovery Anomaly
         Japan         33320438         72997                0 FLAGGED — Zero Recovery Anomaly
  Korea, South         30615522         34093                0 FLAGGED — Zero Recovery Anomaly
         Italy         25603510        188322                0 FLAGGED — Zero Recovery Anomaly
United Kingdom         24658705        220721                0 FLAGGED — Zero Recovery Anomaly

In [18]:
# Reconnect to database
conn = sqlite3.connect(db_path)

query5_fixed = """
SELECT
    YearMonth,
    Year,
    Month,
    ROUND(SUM(Daily_New_Cases), 0)        AS Monthly_New_Cases,
    ROUND(AVG(Daily_New_Cases), 0)        AS Avg_Daily_New_Cases,
    ROUND(MAX(Daily_New_Cases), 0)        AS Peak_Day_Cases
FROM (
    SELECT
        Date,
        Year,
        Month,
        YearMonth,
        SUM(Confirmed) - LAG(SUM(Confirmed)) OVER (ORDER BY Date) AS Daily_New_Cases
    FROM covid_data
    GROUP BY Date, Year, Month, YearMonth
)
WHERE Daily_New_Cases >= 0
GROUP BY YearMonth, Year, Month
ORDER BY YearMonth
"""

monthly = pd.read_sql(query5_fixed, conn)
print("=" * 60)
print("  SQL QUERY 5 (FIXED) — MONTHLY WAVE ANALYSIS")
print("=" * 60)
print(monthly.to_string(index=False))
monthly.to_csv(f'{base}/data/sql/sql_query5_monthly_waves.csv', index=False)

conn.close()
print("\nFixed. Saved: sql_query5_monthly_waves.csv")

  SQL QUERY 5 (FIXED) — MONTHLY WAVE ANALYSIS
YearMonth  Year  Month  Monthly_New_Cases  Avg_Daily_New_Cases  Peak_Day_Cases
  2020-01  2020      1           9,370.00             1,041.00        2,651.00
  2020-02  2020      2          76,096.00             2,624.00       15,152.00
  2020-03  2020      3         783,348.00            25,269.00       78,442.00
  2020-04  2020      4       2,412,716.00            80,424.00      118,581.00
  2020-05  2020      5       2,901,229.00            93,588.00      135,043.00
  2020-06  2020      6       4,292,072.00           143,069.00      192,490.00
  2020-07  2020      7       7,118,656.00           229,634.00      287,462.00
  2020-08  2020      8       7,941,296.00           256,171.00      307,073.00
  2020-09  2020      9       8,498,335.00           283,278.00      330,614.00
  2020-10  2020     10      12,122,070.00           391,035.00      571,742.00
  2020-11  2020     11      17,266,494.00           575,550.00      695,342.00
  2020

In [19]:
excel_path = f'{base}/excel_exports/Pandemic_Data_For_Dashboard.xlsx'

# Reload all query results fresh
conn    = sqlite3.connect(db_path)
kpi     = pd.read_sql(query1,       conn)
top10   = pd.read_sql(query2,       conn)
yearly  = pd.read_sql(query3,       conn)
anomaly = pd.read_sql(query4,       conn)
monthly = pd.read_sql(query5_fixed, conn)
conn.close()

issues_df    = pd.read_csv(f'{base}/data/quality_reports/data_quality_issues.csv')
completeness = pd.read_csv(f'{base}/data/quality_reports/completeness_scores.csv')

# Build a clean global trend table for Excel charts
global_trend = (
    df.groupby('Date')[['Confirmed','Deaths','Recovered','Active']]
    .sum()
    .reset_index()
)
global_trend['Daily_New_Cases'] = global_trend['Confirmed'].diff().clip(lower=0).round(0)
global_trend['Rolling_7Day']    = global_trend['Daily_New_Cases'].rolling(7).mean().round(0)
global_trend['Date']            = global_trend['Date'].dt.strftime('%Y-%m-%d')

# Build country-level summary (one row per country, latest snapshot)
country_summary = (
    df[df['Date'] == df['Date'].max()]
    [['Country','Confirmed','Deaths','Recovered',
      'Active','Fatality_Rate','Recovery_Rate']]
    .sort_values('Confirmed', ascending=False)
    .reset_index(drop=True)
)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    kpi.to_excel(            writer, sheet_name='KPI_Summary',       index=False)
    global_trend.to_excel(   writer, sheet_name='Global_Trend',      index=False)
    monthly.to_excel(        writer, sheet_name='Monthly_Waves',      index=False)
    top10.to_excel(          writer, sheet_name='Top10_Countries',    index=False)
    country_summary.to_excel(writer, sheet_name='All_Countries',      index=False)
    yearly.to_excel(         writer, sheet_name='Yearly_Trend',       index=False)
    anomaly.to_excel(        writer, sheet_name='Quality_Flags',      index=False)
    issues_df.to_excel(      writer, sheet_name='Issues_Log',         index=False)
    completeness.to_excel(   writer, sheet_name='Completeness_Score', index=False)

print("Excel file exported successfully!")
print(f"Location : {excel_path}")
print(f"\n9 sheets ready for dashboard building:")
print("  1. KPI_Summary        → Global KPI scorecards")
print("  2. Global_Trend       → Line chart (confirmed/deaths over time)")
print("  3. Monthly_Waves      → Bar/line chart (wave analysis)")
print("  4. Top10_Countries    → Bar chart (top 10 countries)")
print("  5. All_Countries      → Full country table + map chart")
print("  6. Yearly_Trend       → Column chart (year over year)")
print("  7. Quality_Flags      → Data quality anomaly table")
print("  8. Issues_Log         → Full quality audit log")
print("  9. Completeness_Score → Country completeness table")

Excel file exported successfully!
Location : /content/drive/MyDrive/Pandemic_Dashboard_Project/excel_exports/Pandemic_Data_For_Dashboard.xlsx

9 sheets ready for dashboard building:
  1. KPI_Summary        → Global KPI scorecards
  2. Global_Trend       → Line chart (confirmed/deaths over time)
  3. Monthly_Waves      → Bar/line chart (wave analysis)
  4. Top10_Countries    → Bar chart (top 10 countries)
  5. All_Countries      → Full country table + map chart
  6. Yearly_Trend       → Column chart (year over year)
  7. Quality_Flags      → Data quality anomaly table
  8. Issues_Log         → Full quality audit log
  9. Completeness_Score → Country completeness table
